# Extract Routing Trace from TinyMoE-100m-2x8

**v3: Add full trace analysis**

- Count expert frequency per layer
- Calculate entropy (0 = single expert, 1.0 = uniform)
- Calculate p_adjacent (reuse between adjacent tokens)
- Calculate p_window_10 (reuse in 10-token window)

Expected output:
```
Total records: 10000
entropy = 0.5-0.8 (realistic MoE) or 0.0-0.3 (degenerate)
p_adjacent = 0.7-0.9 (high reuse)
p_window_10 = 0.9-0.99 (very high reuse)
```

In [ ]:
# Install Rust 1.93.0
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --default-toolchain 1.93.0
!rustup default 1.93.0
!rustc --version

In [ ]:
# Install dependencies for trace extraction
!pip install torch transformers accelerate

In [ ]:
# ============================================================
# extract_routing_trace.py — Colab-ready (v2, single forward)
# ============================================================

import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig

MODEL_ID = "FlameF0X/TinyMoE-100m-2x8"
OUT_PATH = "/content/routing-trace.jsonl"

print(f"Loading {MODEL_ID}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float32,
    device_map="auto",
)
model.eval()

config = model.config
print(f"num_hidden_layers = {config.num_hidden_layers}")
print(f"num_experts = {getattr(config, 'num_local_experts', getattr(config, 'num_experts', '?'))}")
print(f"num_experts_per_tok = {getattr(config, 'num_experts_per_tok', '?')}")

trace = []

def make_hook(layer_idx):
    def hook(module, args, output):
        if isinstance(output, tuple) and len(output) >= 3:
            _, top_k_weights, top_k_index = output[0], output[1], output[2]
        elif isinstance(output, tuple) and len(output) == 2:
            top_k_index = output[1]
            top_k_weights = None
        else:
            router_logits = output if not isinstance(output, tuple) else output[0]
            top_k = getattr(module, "top_k", 2)
            top_k_weights, top_k_index = torch.topk(
                torch.softmax(router_logits, dim=-1), top_k, dim=-1
            )

        if top_k_index.dim() == 3:
            top_k_index = top_k_index.reshape(-1, top_k_index.shape[-1])
        if top_k_weights is not None and top_k_weights.dim() == 3:
            top_k_weights = top_k_weights.reshape(-1, top_k_weights.shape[-1])

        n_tokens = top_k_index.shape[0]

        if layer_idx == 0:
            print(f"[hook] layer={layer_idx} index_shape={tuple(top_k_index.shape)}")

        for pos in range(n_tokens):
            exp_list = sorted(top_k_index[pos].tolist())
            w_list = (
                top_k_weights[pos].tolist()
                if top_k_weights is not None
                else [None] * len(exp_list)
            )
            trace.append({
                "layer": layer_idx,
                "pos": pos,
                "experts": exp_list,
                "weights": w_list,
            })

    return hook

hooks = []
for name, module in model.named_modules():
    if name.endswith(".mlp.gate"):
        parts = name.split(".")
        layer_idx = None
        for i, p in enumerate(parts):
            if p == "layers" and i + 1 < len(parts):
                try:
                    layer_idx = int(parts[i + 1])
                except ValueError:
                    pass
                break
        if layer_idx is None:
            continue
        h = module.register_forward_hook(make_hook(layer_idx))
        hooks.append(h)
        print(f"Hooked layer {layer_idx}: {name}")

print(f"Total hooks: {len(hooks)}")

seed_text = (
    "The quick brown fox jumps over the lazy dog. "
    "Machine learning models process text token by token. "
    "Mixture of experts routes each token to specialized subnetworks. "
    "Storage layouts affect memory access patterns. "
    "Cold reads hit the disk, warm reads hit the cache. "
) * 20

inputs = tokenizer(
    seed_text,
    return_tensors="pt",
    truncation=True,
    max_length=1000,
).to(model.device)

n_input = inputs["input_ids"].shape[1]
print(f"Input sequence length: {n_input}")

print("Running single forward pass...")
with torch.no_grad():
    _ = model(**inputs)

for h in hooks:
    h.remove()
print("Hooks removed.")
print(f"Trace size: {len(trace)} records")
print(f"Expected: {n_input} tokens x {len(hooks)} layers = {n_input * len(hooks)}")

trace.sort(key=lambda r: (r["layer"], r["pos"]))

with open(OUT_PATH, "w") as f:
    for rec in trace:
        f.write(json.dumps(rec) + "\n")

print(f"Wrote {len(trace)} records to {OUT_PATH}")

In [ ]:
# ============================================================
# Full trace analysis
# ============================================================

import json
from collections import Counter
import math

with open("/content/routing-trace.jsonl") as f:
    trace = [json.loads(line) for line in f]

print(f"Total records: {len(trace)}")
print(f"Layers: {sorted(set(r['layer'] for r in trace))}")
print(f"Tokens per layer: {len(trace) // 10}")

# Expert frequency per layer
per_layer = {}
for r in trace:
    per_layer.setdefault(r["layer"], Counter()).update(r["experts"])

for layer in sorted(per_layer):
    c = per_layer[layer]
    total = sum(c.values())
    top = c.most_common(8)
    entropy = -sum((v/total) * math.log(v/total) for v in c.values() if v > 0)
    max_entropy = math.log(8)
    print(f"layer {layer}: {top}")
    print(f"  entropy = {entropy:.3f} / {max_entropy:.3f} ({entropy/max_entropy*100:.1f}%)")

# Reuse between adjacent tokens (p_adjacent)
for layer in sorted(per_layer):
    layer_recs = sorted([r for r in trace if r["layer"] == layer], key=lambda x: x["pos"])
    reuse_count = 0
    for i in range(1, len(layer_recs)):
        prev = set(layer_recs[i-1]["experts"])
        curr = set(layer_recs[i]["experts"])
        if prev & curr:
            reuse_count += 1
    p_adjacent = reuse_count / max(1, len(layer_recs) - 1)
    print(f"layer {layer}: p_adjacent = {p_adjacent:.3f}")

# Reuse in 10-token window (p_window_10)
for layer in sorted(per_layer):
    layer_recs = sorted([r for r in trace if r["layer"] == layer], key=lambda x: x["pos"])
    window = 10
    total = 0
    reuse = 0
    for i in range(len(layer_recs)):
        lo = max(0, i - window)
        recent = set()
        for j in range(lo, i):
            recent.update(layer_recs[j]["experts"])
        curr = set(layer_recs[i]["experts"])
        total += 1
        if recent & curr:
            reuse += 1
    print(f"layer {layer}: p_window_10 = {reuse / total:.3f}")

In [ ]:
# Download the trace
try:
    from google.colab import files
    files.download(OUT_PATH)
except ImportError:
    print(f"(not in Colab; file at {OUT_PATH})")